# Semantic Geometry — Hidden State Extraction

Extracts hidden states from all layers of a language model for both the multilingual and constraints datasets.

**Outputs per model:**
- `{model}_multilingual_last.npy` — last-token pooling, shape `(1200, n_layers, hidden_dim)`
- `{model}_multilingual_mean.npy` — mean pooling, shape `(1200, n_layers, hidden_dim)`
- `{model}_constraints_last.npy` — last-token pooling, shape `(3840, n_layers, hidden_dim)`
- `{model}_constraints_mean.npy` — mean pooling, shape `(3840, n_layers, hidden_dim)`
- `{model}_metadata.json` — model info, sentence order, normalised depth map

**Resilience:** checkpoints after every batch, resumes on crash.

## 0. Setup

In [ ]:
# === CONFIGURATION — edit this cell ===

MODEL_KEY = "llama3.2-3b"  # one of the keys in MODEL_CONFIGS below
BATCH_SIZE = 4              # 4-8 for Colab T4; 1 for Mac MPS
HF_TOKEN = ""  # needed for Llama (https://huggingface.co/settings/tokens)

# Where to save outputs (Google Drive for persistence)
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/semantic_geometry"

# Precision notes:
#   bfloat16 — used for all models.  Same memory as float16 but 8× the dynamic range,
#               which prevents overflow/underflow in the forward pass.  Geometry analyses
#               (eigendecomposition, Mahalanobis) are done on the float32 values that the
#               extraction engine writes to disk (it always calls .float() before saving).
#   float16   — NOT used: limited range causes silent NaN / inf in deep layers.

MODEL_CONFIGS = {
    # Extracted with this notebook on a Colab GPU. Gemma 4 E4B was extracted with
    # extraction/extract_activations.py (Apple MPS).
    "llama3.2-3b": {
        "hf_id": "meta-llama/Llama-3.2-3B",
        "dtype": "bfloat16",
    },
    "qwen2.5-3b": {
        "hf_id": "Qwen/Qwen2.5-3B",
        "dtype": "bfloat16",
    },
    "pythia-2.8b": {
        "hf_id": "EleutherAI/pythia-2.8b",
        "dtype": "bfloat16",
    },
}

In [ ]:
# === Install / upgrade dependencies ===
# Upgrading both transformers AND huggingface_hub together prevents two known Colab errors:
#   - StrictDataclassDefinitionError (version mismatch affecting some model configs)
#   - KeyError: 'gemma4' / ValueError: unrecognised architecture (old transformers < 5)
#
# IMPORTANT: After running this cell for the first time in a new session, go to
#   Runtime → Restart session, then run all cells again from the top.
!pip install -q --upgrade transformers huggingface_hub accelerate torch numpy

In [ ]:
# === Mount Drive & create output dir ===
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = os.path.join(DRIVE_DIR, MODEL_KEY)
else:
    OUTPUT_DIR = f"/content/outputs/{MODEL_KEY}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# === HF login (if needed for gated models like Llama) ===
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace.")
else:
    print("No HF token set. Skipping login (ok for non-gated models).")

## 1. Upload datasets

Upload `multilingual.jsonl` and `constraints_sentences.jsonl` from `data/final/`.

Either:
- Use the file upload button in Colab's sidebar, or
- Copy from Drive if you've already uploaded them there, or
- Run the cell below to upload interactively.

In [ ]:
# Option A: upload from local machine
# from google.colab import files
# uploaded = files.upload()  # select multilingual.jsonl and constraints_sentences.jsonl

# Option B: copy from Drive (if you already put them there)
# !cp /content/drive/MyDrive/semantic_geometry/data/multilingual.jsonl /content/
# !cp /content/drive/MyDrive/semantic_geometry/data/constraints_sentences.jsonl /content/

# Option C: set paths directly if files are already available
MULTILINGUAL_PATH = "/content/multilingual.jsonl"
CONSTRAINTS_PATH = "/content/constraints_sentences.jsonl"

# Verify
for p in [MULTILINGUAL_PATH, CONSTRAINTS_PATH]:
    if os.path.exists(p):
        size = os.path.getsize(p)
        print(f"  {p}: {size:,} bytes")
    else:
        print(f"  {p}: NOT FOUND — upload it before proceeding")

## 2. Load datasets & build sentence lists

In [ ]:
import json

def load_multilingual(path):
    """Load multilingual JSONL → list of sentence dicts with metadata."""
    sentences = []
    with open(path) as f:
        for line in f:
            record = json.loads(line)
            for s in record['sentences']:
                sentences.append({
                    'text': s['text'],
                    'type': s['type'],
                    'factor_id': record['factor_id'],
                    'language': record['language'],
                    'domain': record['domain'],
                    'adjective': record['adjective'],
                })
    return sentences

def load_constraints(path):
    """Load constraints flat JSONL → list of sentence dicts with metadata."""
    sentences = []
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            sentences.append({
                'text': row['sentence'],
                'chain_id': row['chain_id'],
                'arm': row['arm'],
                'register': row['register'],
                'level': row['level'],
                'concept': row['concept'],
            })
    return sentences

ml_sentences = load_multilingual(MULTILINGUAL_PATH)
ct_sentences = load_constraints(CONSTRAINTS_PATH)

print(f"Multilingual: {len(ml_sentences)} sentences")
print(f"Constraints:  {len(ct_sentences)} sentences")
print(f"Total:        {len(ml_sentences) + len(ct_sentences)} sentences")
print(f"\nFirst ML: {ml_sentences[0]['text'][:60]}")
print(f"First CT: {ct_sentences[0]['text'][:60]}")

## 3. Load model

In [ ]:
import torch
import time

# ── Workaround: SiglipTextConfig @strict bug in transformers 5.x ──────────────
# transformers 5.x decorates SiglipTextConfig with @strict from huggingface_hub,
# but SiglipTextConfig is not a dataclass.  This causes StrictDataclassDefinitionError
# for EVERY AutoModel call (the auto registry lazily imports all config modules).
# Making @strict a no-op patches this without affecting correctness.
try:
    import huggingface_hub.dataclasses as _hfdc
    if hasattr(_hfdc, "strict"):
        _hfdc.strict = lambda cls=None, accept_kwargs=False: (
            (lambda c: c) if cls is None else cls
        )
except Exception as _e:
    print(f"[info] @strict patch skipped: {_e}")
# ─────────────────────────────────────────────────────────────────────────────

config = MODEL_CONFIGS[MODEL_KEY]
hf_id = config["hf_id"]
dtype_str = config["dtype"]
dtype_map = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
model_dtype = dtype_map[dtype_str]

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Model:  {hf_id}")
print(f"Dtype:  {dtype_str}")
print(f"Device: {device}")

# Load
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(hf_id, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    hf_id,
    dtype=model_dtype,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)
model.eval()
model.to(device)

# Probe to get n_layers / hidden_dim (handles nested configs like Gemma 4 PLE)
_probe_input = tokenizer("hello", return_tensors="pt").to(device)
with torch.no_grad():
    _probe_out = model(**_probe_input, output_hidden_states=True)
_hs = _probe_out.hidden_states
n_layers = len(_hs)
hidden_dim = _hs[0].shape[-1]
del _probe_out, _probe_input, _hs

print(f"\nLoaded in {time.time()-t0:.1f}s")
print(f"Layers:     {n_layers} (1 embedding + {n_layers-1} transformer)")
print(f"Hidden dim: {hidden_dim}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

# GPU memory
if device.type == 'cuda':
    mem = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory: {mem:.2f} GB")

## 4. Extraction engine

In [ ]:
import numpy as np
import gc

def extract_hidden_states(sentences, output_prefix, batch_size=BATCH_SIZE):
    """
    Extract hidden states for a list of sentences.
    Saves both last-token and mean-pooled representations.
    
    Memory-safe: writes each batch directly to disk via memory-mapped arrays.
    Nothing accumulates in RAM beyond one batch at a time.
    Checkpoints after every batch — survives crashes.
    
    Args:
        sentences: list of dicts with 'text' key
        output_prefix: e.g., '/content/drive/.../llama3.2-3b_multilingual'
        batch_size: sentences per forward pass
    
    Saves:
        {output_prefix}_last.npy  — shape (N, n_layers, hidden_dim)
        {output_prefix}_mean.npy  — shape (N, n_layers, hidden_dim)
    """
    N = len(sentences)
    checkpoint_path = f"{output_prefix}_checkpoint.json"
    last_path = f"{output_prefix}_last.npy"
    mean_path = f"{output_prefix}_mean.npy"
    
    # --- Resume from checkpoint if exists ---
    start_idx = 0
    skipped = []
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            ckpt = json.load(f)
        start_idx = ckpt['last_completed_index'] + 1
        skipped = ckpt.get('skipped', [])
        print(f"  Resuming from sentence {start_idx}/{N} (skipped: {len(skipped)})")
    
    # --- Create or open memory-mapped arrays on disk ---
    # These live on disk, not in RAM. Only the current batch is in memory.
    if start_idx == 0 or not os.path.exists(last_path):
        # Create new files
        acts_last = np.lib.format.open_memmap(
            last_path, mode='w+', dtype=np.float32, shape=(N, n_layers, hidden_dim))
        acts_mean = np.lib.format.open_memmap(
            mean_path, mode='w+', dtype=np.float32, shape=(N, n_layers, hidden_dim))
        print(f"  Created memmap arrays: ({N}, {n_layers}, {hidden_dim}) "
              f"= {N * n_layers * hidden_dim * 4 / 1e6:.1f} MB each on disk")
    else:
        # Resume into existing files
        acts_last = np.lib.format.open_memmap(last_path, mode='r+')
        acts_mean = np.lib.format.open_memmap(mean_path, mode='r+')
        print(f"  Opened existing memmap arrays: shape={acts_last.shape}")
    
    # --- Extract ---
    t0 = time.time()
    
    for batch_start in range(start_idx, N, batch_size):
        batch_end = min(batch_start + batch_size, N)
        batch_texts = [sentences[i]['text'] for i in range(batch_start, batch_end)]
        
        try:
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            ).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs, output_hidden_states=True)
            
            hidden_states = outputs.hidden_states  # tuple of tensors, each (B, T, H)
            attention_mask = inputs['attention_mask']  # (B, T)
            B = attention_mask.shape[0]
            
            for layer_idx, hs in enumerate(hidden_states):
                if layer_idx >= n_layers:
                    break
                hs_f = hs.float()  # (B, T, H) in float32
                
                # --- Last-token pooling ---
                lengths = attention_mask.sum(dim=1) - 1
                last_token = hs_f[torch.arange(B, device=device), lengths]
                acts_last[batch_start:batch_end, layer_idx] = last_token.cpu().numpy()
                
                # --- Mean pooling ---
                mask = attention_mask.unsqueeze(-1).float()
                summed = (hs_f * mask).sum(dim=1)
                counts = mask.sum(dim=1).clamp(min=1e-6)
                mean_pooled = summed / counts
                acts_mean[batch_start:batch_end, layer_idx] = mean_pooled.cpu().numpy()
            
            # === CRITICAL: free ALL GPU memory immediately ===
            del outputs, hidden_states, inputs, hs, hs_f, last_token, mean_pooled, summed, mask, counts
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            elif device.type == 'mps':
                torch.mps.empty_cache()
            gc.collect()
        
        except Exception as e:
            print(f"  ERROR at batch {batch_start}-{batch_end}: {e}")
            for i in range(batch_start, batch_end):
                skipped.append(i)
            # Still try to free memory after error
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            continue
        
        # --- Flush memmap to disk after every batch ---
        acts_last.flush()
        acts_mean.flush()
        
        # --- Checkpoint after every batch ---
        with open(checkpoint_path, 'w') as f:
            json.dump({
                'last_completed_index': batch_end - 1,
                'total_sentences': N,
                'skipped': skipped,
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            }, f)
        
        # --- Progress ---
        batch_num = (batch_start - start_idx) // batch_size
        if (batch_num + 1) % 10 == 0 or batch_end == N:
            elapsed = time.time() - t0
            done = batch_end - start_idx
            total = N - start_idx
            eta = (elapsed / done) * (total - done) if done > 0 else 0
            mem_info = ""
            if device.type == 'cuda':
                mem_info = f"  GPU={torch.cuda.memory_allocated()/1e9:.1f}GB"
            print(f"  {batch_end:5d}/{N} ({100*batch_end/N:.1f}%)  "
                  f"elapsed={elapsed:.0f}s  ETA={eta:.0f}s  "
                  f"skipped={len(skipped)}{mem_info}")
    
    # --- Close memmap (data already on disk) ---
    del acts_last, acts_mean
    
    # Remove checkpoint (extraction complete)
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
    
    elapsed = time.time() - t0
    file_size = os.path.getsize(last_path) / 1e6
    print(f"  Done. {N} sentences in {elapsed:.1f}s. Skipped: {len(skipped)}")
    print(f"  Last-token: {last_path} ({file_size:.1f} MB)")
    print(f"  Mean-pool:  {mean_path} ({file_size:.1f} MB)")
    
    return skipped

## 5. Run extraction

In [ ]:
print("=" * 60)
print(f"EXTRACTING: {MODEL_KEY} ({hf_id})")
print(f"Device: {device} | Dtype: {dtype_str} | Batch: {BATCH_SIZE}")
print(f"Layers: {n_layers} | Hidden: {hidden_dim}")
print("=" * 60)

# --- Multilingual ---
print(f"\n--- Multilingual ({len(ml_sentences)} sentences) ---")
ml_prefix = os.path.join(OUTPUT_DIR, f"{MODEL_KEY}_multilingual")
ml_skipped = extract_hidden_states(ml_sentences, ml_prefix)

# --- Constraints ---
print(f"\n--- Constraints ({len(ct_sentences)} sentences) ---")
ct_prefix = os.path.join(OUTPUT_DIR, f"{MODEL_KEY}_constraints")
ct_skipped = extract_hidden_states(ct_sentences, ct_prefix)

## 6. Save metadata

In [ ]:
# Normalised depth: layer_i / (n_layers - 1) so all models share [0, 1] x-axis
normalised_depth = [round(i / (n_layers - 1), 4) for i in range(n_layers)]

metadata = {
    "model_key": MODEL_KEY,
    "hf_id": hf_id,
    "num_layers": n_layers,
    "hidden_dim": hidden_dim,
    "dtype": dtype_str,
    "device": str(device),
    "pooling_methods": ["last_token", "mean"],
    "normalised_depth": normalised_depth,
    "multilingual": {
        "num_sentences": len(ml_sentences),
        "skipped": ml_skipped,
        "sentence_order": [s['text'][:50] for s in ml_sentences],
        "factor_ids": [s['factor_id'] for s in ml_sentences],
        "languages": [s['language'] for s in ml_sentences],
        "types": [s['type'] for s in ml_sentences],
    },
    "constraints": {
        "num_sentences": len(ct_sentences),
        "skipped": ct_skipped,
        "sentence_order": [s['text'][:50] for s in ct_sentences],
        "chain_ids": [s['chain_id'] for s in ct_sentences],
        "arms": [s['arm'] for s in ct_sentences],
        "levels": [s['level'] for s in ct_sentences],
    },
    "extraction_date": time.strftime('%Y-%m-%d %H:%M:%S'),
}

meta_path = os.path.join(OUTPUT_DIR, f"{MODEL_KEY}_metadata.json")
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved: {meta_path}")

## 7. Verify extraction

In [ ]:
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

for dataset_name in ['multilingual', 'constraints']:
    print(f"\n--- {dataset_name} ---")
    for pooling in ['last', 'mean']:
        path = os.path.join(OUTPUT_DIR, f"{MODEL_KEY}_{dataset_name}_{pooling}.npy")
        acts = np.load(path)
        
        print(f"  {pooling:5s}: shape={acts.shape}  dtype={acts.dtype}  "
              f"size={acts.nbytes/1e6:.1f}MB")
        
        # NaN/Inf check
        n_nan = np.isnan(acts).sum()
        n_inf = np.isinf(acts).sum()
        print(f"         NaN={n_nan}  Inf={n_inf}  {'PASS' if n_nan==0 and n_inf==0 else 'FAIL'}")
        
        # Sanity: cosine similarity between sentence 0 and 1 at each layer
        from numpy.linalg import norm
        a, b = acts[0], acts[1]
        cos = np.array([np.dot(a[l], b[l]) / (norm(a[l]) * norm(b[l]) + 1e-10) 
                        for l in range(acts.shape[1])])
        print(f"         cos(s0,s1): layer0={cos[0]:.4f}  mid={cos[len(cos)//2]:.4f}  "
              f"last={cos[-1]:.4f}")

print(f"\nNormalised depth range: {normalised_depth[0]} → {normalised_depth[-1]}")
print(f"\nAll files in {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:50s} {size/1e6:8.1f} MB")

## 8. Free GPU and next model

To extract another model in the same session, run this cell to free memory, then go back to cell 0, change `MODEL_KEY`, and re-run from cell 3 (Load model).

In [ ]:
# Free model from GPU
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("Model freed. Change MODEL_KEY and re-run from cell 3.")